<a href="https://colab.research.google.com/github/juanarrow/BigData-notebooks/blob/main/sesion-23.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 23 - Computación Distribuida

**Módulo:** Big Data Aplicado (BDA - 5075)

---

## Objetivos

- Comprender la diferencia entre computación paralela y distribuida
- Simular paralelismo con `multiprocessing` y `concurrent.futures`
- Medir el speedup real y compararlo con la Ley de Amdahl
- Visualizar el impacto del overhead de comunicación

---

## 1. Secuencial vs Paralelo: primera comparación

Vamos a simular una tarea costosa (cálculo de números primos) y comparar ejecución secuencial vs paralela.

In [ ]:
import time
import math
from multiprocessing import Pool, cpu_count

def es_primo(n):
    """Determina si n es primo (función costosa para números grandes)."""
    if n < 2:
        return False
    if n < 4:
        return True
    if n % 2 == 0 or n % 3 == 0:
        return False
    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True

numeros = list(range(100_000, 200_000))
print(f"Números a evaluar: {len(numeros):,}")
print(f"CPUs disponibles: {cpu_count()}")

inicio = time.time()
resultado_seq = [es_primo(n) for n in numeros]
tiempo_seq = time.time() - inicio
primos_encontrados = sum(resultado_seq)
print(f"\nSECUENCIAL: {tiempo_seq:.2f}s ({primos_encontrados} primos encontrados)")

for workers in [2, 4, 8]:
    w = min(workers, cpu_count())
    inicio = time.time()
    with Pool(w) as pool:
        resultado_par = pool.map(es_primo, numeros)
    tiempo_par = time.time() - inicio
    speedup = tiempo_seq / tiempo_par
    eficiencia = speedup / w * 100
    print(f"PARALELO ({w} workers): {tiempo_par:.2f}s → Speedup: {speedup:.2f}x, Eficiencia: {eficiencia:.0f}%")

---

## 2. Ley de Amdahl: speedup teórico vs real

La Ley de Amdahl predice el speedup máximo según la fracción secuencial del programa:

$$Speedup = \frac{1}{S + \frac{1-S}{N}}$$

Donde **S** es la fracción secuencial y **N** el número de procesadores.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def amdahl_speedup(s, n):
    """Calcula el speedup según la Ley de Amdahl."""
    return 1.0 / (s + (1 - s) / n)

procesadores = np.arange(1, 65)
fracciones_seq = [0.0, 0.05, 0.10, 0.25, 0.50]
colores = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']

plt.figure(figsize=(12, 6))

for s, color in zip(fracciones_seq, colores):
    speedups = [amdahl_speedup(s, n) for n in procesadores]
    plt.plot(procesadores, speedups, color=color, linewidth=2,
             label=f'S = {s*100:.0f}% (máx: {1/s:.1f}x)' if s > 0 else f'S = 0% (ideal)')

plt.xlabel('Número de procesadores (N)', fontsize=12)
plt.ylabel('Speedup', fontsize=12)
plt.title('Ley de Amdahl: Speedup teórico según fracción secuencial', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.xlim(1, 64)
plt.ylim(0, 25)
plt.show()

print("Conclusión: con solo un 10% de código secuencial,")
print("el speedup NUNCA superará 10x, aunque tengas 1000 procesadores.")

---

## 3. Midiendo la fracción secuencial real

Vamos a crear una tarea con una fracción secuencial conocida y verificar si las mediciones coinciden con Amdahl.

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor

def tarea_paralela(n):
    """Simula trabajo pesado paralelizable."""
    total = 0
    for i in range(n):
        total += math.sqrt(i) * math.sin(i)
    return total

def ejecutar_con_fraccion_secuencial(datos, frac_seq, num_workers):
    """Ejecuta una tarea con una parte secuencial forzada."""
    n_total = len(datos)
    n_seq = int(n_total * frac_seq)
    n_par = n_total - n_seq

    inicio = time.time()

    for d in datos[:n_seq]:
        tarea_paralela(d)

    if n_par > 0:
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            list(executor.map(tarea_paralela, datos[n_seq:]))

    return time.time() - inicio

datos = [50_000] * 40

t_base = ejecutar_con_fraccion_secuencial(datos, frac_seq=0.0, num_workers=1)
print(f"Tiempo base (1 worker, 0% seq): {t_base:.2f}s\n")

print(f"{'Frac. Seq':>10} {'Workers':>8} {'Tiempo':>8} {'Speedup Real':>13} {'Speedup Amdahl':>15}")
print("-" * 60)

resultados_exp = []

for frac in [0.0, 0.1, 0.25, 0.5]:
    for w in [1, 2, 4]:
        w_real = min(w, cpu_count())
        t = ejecutar_con_fraccion_secuencial(datos, frac_seq=frac, num_workers=w_real)
        speedup_real = t_base / t
        speedup_teorico = amdahl_speedup(frac, w_real)
        resultados_exp.append((frac, w_real, speedup_real, speedup_teorico))
        print(f"{frac*100:>9.0f}% {w_real:>8} {t:>7.2f}s {speedup_real:>12.2f}x {speedup_teorico:>14.2f}x")

---

## 4. Overhead de comunicación

El paralelismo no es gratis: crear procesos, distribuir datos y recoger resultados tiene un coste. Para tareas **muy pequeñas**, el overhead supera al beneficio.

In [ ]:
import time
from multiprocessing import Pool

def tarea_ligera(x):
    return x * 2

def tarea_pesada(x):
    total = 0
    for i in range(100_000):
        total += math.sqrt(i + x)
    return total

tamanos = [10, 100, 1_000, 10_000]

print("=" * 70)
print("TAREA LIGERA (x * 2): el overhead domina")
print("=" * 70)

for n in tamanos:
    datos = list(range(n))

    inicio = time.time()
    [tarea_ligera(x) for x in datos]
    t_seq = time.time() - inicio

    inicio = time.time()
    with Pool(4) as pool:
        pool.map(tarea_ligera, datos)
    t_par = time.time() - inicio

    ganancia = "más rápido" if t_par < t_seq else "MÁS LENTO"
    print(f"n={n:>6,}: Seq={t_seq:.4f}s, Par={t_par:.4f}s → Paralelo {ganancia}")

print(f"\n{'=' * 70}")
print("TAREA PESADA (100K operaciones): el paralelismo compensa")
print("=" * 70)

for n in [4, 8, 16, 32]:
    datos = list(range(n))

    inicio = time.time()
    [tarea_pesada(x) for x in datos]
    t_seq = time.time() - inicio

    inicio = time.time()
    with Pool(4) as pool:
        pool.map(tarea_pesada, datos)
    t_par = time.time() - inicio

    speedup = t_seq / t_par
    print(f"n={n:>3}: Seq={t_seq:.2f}s, Par={t_par:.2f}s → Speedup: {speedup:.2f}x")

---

## 5. Simulación de un sistema distribuido simple

Simulamos un modelo maestro-trabajador donde un coordinador reparte tareas entre workers.

In [ ]:
import random
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

def worker_procesar_bloque(args):
    """Simula un worker procesando un bloque de datos."""
    worker_id, bloque = args
    inicio = time.time()

    resultado = sum(x ** 2 for x in bloque)

    latencia = random.uniform(0.01, 0.05)
    time.sleep(latencia)

    tiempo = time.time() - inicio
    return worker_id, resultado, tiempo

datos_totales = list(range(1, 100_001))
num_workers = 4
tam_bloque = len(datos_totales) // num_workers

bloques = []
for i in range(num_workers):
    inicio_bloque = i * tam_bloque
    fin_bloque = inicio_bloque + tam_bloque if i < num_workers - 1 else len(datos_totales)
    bloques.append((i, datos_totales[inicio_bloque:fin_bloque]))

print(f"Datos totales: {len(datos_totales):,} elementos")
print(f"Workers: {num_workers}")
print(f"Tamaño de bloque: ~{tam_bloque:,} elementos")
print()

inicio_total = time.time()
resultado_seq = sum(x ** 2 for x in datos_totales)
tiempo_seq = time.time() - inicio_total
print(f"SECUENCIAL: resultado = {resultado_seq:,}, tiempo = {tiempo_seq:.4f}s")

inicio_total = time.time()
resultados_parciales = []

with ProcessPoolExecutor(max_workers=num_workers) as executor:
    futuros = {executor.submit(worker_procesar_bloque, b): b[0] for b in bloques}

    for futuro in as_completed(futuros):
        wid, res, t = futuro.result()
        resultados_parciales.append(res)
        print(f"  Worker {wid} completó en {t:.4f}s")

resultado_dist = sum(resultados_parciales)
tiempo_dist = time.time() - inicio_total

print(f"\nDISTRIBUIDO: resultado = {resultado_dist:,}, tiempo = {tiempo_dist:.4f}s")
print(f"Speedup: {tiempo_seq / tiempo_dist:.2f}x")
print(f"Resultados coinciden: {resultado_seq == resultado_dist}")

---

## 6. Ley de Gustafson: escalando el problema

A diferencia de Amdahl (problema fijo), Gustafson asume que el problema **crece** con el número de procesadores — más realista en Big Data.

$$Speedup(N) = N - S \times (N - 1)$$

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def gustafson_speedup(s, n):
    return n - s * (n - 1)

procesadores = np.arange(1, 65)
fracciones = [0.05, 0.10, 0.25]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colores = ['#3498db', '#f39c12', '#e74c3c']
for s, color in zip(fracciones, colores):
    amdahl = [amdahl_speedup(s, n) for n in procesadores]
    gustafson = [gustafson_speedup(s, n) for n in procesadores]
    axes[0].plot(procesadores, amdahl, color=color, linewidth=2, label=f'S={s*100:.0f}%')
    axes[1].plot(procesadores, gustafson, color=color, linewidth=2, label=f'S={s*100:.0f}%')

axes[0].set_title('Ley de Amdahl (problema fijo)', fontsize=13)
axes[0].set_xlabel('Procesadores')
axes[0].set_ylabel('Speedup')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 25)

axes[1].set_title('Ley de Gustafson (problema crece)', fontsize=13)
axes[1].set_xlabel('Procesadores')
axes[1].set_ylabel('Speedup')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("En Big Data, Gustafson es más realista:")
print("  → Más datos = más nodos = speedup casi lineal")
print("  → Amdahl pone un techo; Gustafson muestra que el techo sube con el problema")

---

## 7. Efecto de la granularidad de las tareas

¿Es mejor muchas tareas pequeñas o pocas tareas grandes? Experimentemos con distintas granularidades.

In [ ]:
import time
from multiprocessing import Pool

def trabajo_fijo(args):
    """Realiza una cantidad fija de trabajo."""
    iteraciones = args
    total = 0
    for i in range(iteraciones):
        total += math.sqrt(i)
    return total

TRABAJO_TOTAL = 4_000_000
granularidades = [
    ("Muy fina (40K tareas)", 40_000, TRABAJO_TOTAL // 40_000),
    ("Fina (4K tareas)", 4_000, TRABAJO_TOTAL // 4_000),
    ("Media (400 tareas)", 400, TRABAJO_TOTAL // 400),
    ("Gruesa (40 tareas)", 40, TRABAJO_TOTAL // 40),
    ("Muy gruesa (4 tareas)", 4, TRABAJO_TOTAL // 4),
]

print(f"Trabajo total constante: {TRABAJO_TOTAL:,} iteraciones")
print(f"Workers: 4\n")
print(f"{'Granularidad':<25} {'Tareas':>8} {'Iter/tarea':>12} {'Tiempo':>8} {'Speedup':>8}")
print("-" * 65)

inicio = time.time()
trabajo_fijo(TRABAJO_TOTAL)
t_base = time.time() - inicio
print(f"{'Secuencial':<25} {'1':>8} {TRABAJO_TOTAL:>12,} {t_base:>7.3f}s {1.0:>7.2f}x")

for nombre, num_tareas, iter_por_tarea in granularidades:
    tareas = [iter_por_tarea] * num_tareas
    inicio = time.time()
    with Pool(4) as pool:
        pool.map(trabajo_fijo, tareas)
    t = time.time() - inicio
    print(f"{nombre:<25} {num_tareas:>8,} {iter_por_tarea:>12,} {t:>7.3f}s {t_base/t:>7.2f}x")

print("\nConsejos de granularidad:")
print("  → Tareas muy pequeñas: overhead de gestión > beneficio")
print("  → Tareas muy grandes: poco paralelismo, desbalance")
print("  → Ideal: suficientes tareas para mantener todos los workers ocupados")

---

## Recursos

- [Ley de Amdahl - Wikipedia](https://en.wikipedia.org/wiki/Amdahl%27s_law)
- [Python multiprocessing](https://docs.python.org/3/library/multiprocessing.html)
- [concurrent.futures](https://docs.python.org/3/library/concurrent.futures.html)
- [Designing Data-Intensive Applications](https://dataintensive.net/)